# Orbit Wars — Launch Playground

Görsel bir oyun kontrolcüsü. `Playground` sınıfı:
- `planet_table()` ile gezegenleri listeler
- `launch(src_id, tgt_id, ships)` — açıyı **otomatik hesaplar** (`compute_launch`), filo kuyruğa alır
- `step()` — bir tur ilerler
- `render()` — anlık board'u çizer (gezegenler, yörüngeler, güneş, filolar, son atışların hedef noktaları)

Her launch'tan sonra konsola **kaynak → hedef, gemi, açı, ETA, varış turu** yazılır.

In [ ]:
import sys, math, importlib
sys.path.insert(0, '/home/doruk/Documents/Kaggle/Orbit_Wars')

# Reload-friendly imports so edits to the .py modules pick up without restart
import feature_extraction; importlib.reload(feature_extraction)
import navigation;          importlib.reload(navigation)
import playground;          importlib.reload(playground)

from playground import Playground

## 1. Oyunu başlat

Yeni bir 2 oyunculu oyun yarat ve başlangıç durumunu çiz. Her hücreyi yeniden çalıştırmak yeni bir oyun başlatır.

In [ ]:
pg = Playground(num_agents=2)
pg.summary()
pg.render(title=f'Step {pg.step_num} - Initial state')

## 2. Gezegen tablosu

Atış yapacağın `src_id` ve `tgt_id`'leri seçmek için. **ME** = senin gezegenlerin, **Neut** = nötr, **P1** = düşman.
Kind: `static` (sabit), `rot` (yörüngede), `comet` (geçici).

In [ ]:
pg.planet_table()

## 3. Filo gönder

Yukarıdaki tablodan kendi gezegenlerinden (ME) birini seç ve hedef gezegenin id'sini ver.

`compute_launch` arka planda:
- Hareketli hedef için **lead shot** açısı hesaplar.
- Güneşi geçen yörüngeleri reddedip bir sonraki **lap** çözümünü dener.
- Yolda başka gezegene çarpıyorsa o adayı atlar.
- Geçerli yörünge bulamazsa `None` döner ve filo kuyruğa **eklenmez**.

Aynı turda birden fazla `launch()` çağırabilirsin (multi-fleet).

In [ ]:
# ÖRNEK — kendi id'lerinle değiştir
# pg.launch(src_id=0, tgt_id=7, ships=8)
# pg.launch(src_id=0, tgt_id=12, ships=4)

# En yakın hedefe otomatik fırlatma (örnek senaryo):
me = pg.obs['player']
ps = pg.planets()
my_planets = [p for p in ps if p.is_mine(me)]
for src in my_planets:
    targets = [p for p in ps if not p.is_mine(me) and p.id != src.id]
    if not targets:
        continue
    nearest = min(targets, key=lambda t: math.hypot(t.x - src.x, t.y - src.y))
    if src.ships >= nearest.ships + 2:
        pg.launch(src.id, nearest.id, ships=nearest.ships + 2)

pg.render(title=f'Step {pg.step_num} - launches queued (dotted lines)')

## 4. Bir tur ilerle

`pg.step()` env'i bir tur ilerletir; kuyruktaki filolar fiilen oluşur ve hareket eder.
Rakip (`P1`) bu örnekte hiçbir şey yapmıyor (`opponent_action=None` -> boş aksiyon).

In [ ]:
pg.step()
pg.summary()
pg.render()

## 5. Birkaç tur ileri sar

Filoların hedeflere varışını izlemek için art arda step. Aşağıdaki hücreyi tekrar tekrar çalıştırabilirsin.

In [ ]:
for _ in range(5):
    pg.step()
pg.summary()
pg.render(title=f'Step {pg.step_num} - after 5 free turns')

## 6. Tarihçe

Şimdiye kadar yapılan tüm launch'lar.

In [ ]:
for r in pg.history:
    print(f"step {r['step']:>3}  P{r['player']}  "
          f"{r['src_id']:>2} -> {r['tgt_id']:>2}  "
          f"{r['ships']:>3} ships  ETA={r['eta']:>5.2f}  "
          f"arrives~step {r['arrival_step']}")

## 7. Serbest oyun

Aşağıdaki hücreyi düzenle: kendi gezegenlerinden seçtiğin hedeflere `launch` yap, sonra `step()` ile ilerle, `render()` ile gözle.

```python
pg.launch(src_id=..., tgt_id=..., ships=...)
pg.step()
pg.render()
```

In [ ]:
# pg.launch(src_id=..., tgt_id=..., ships=...)
# pg.step()
# pg.render()